# Week 10 Lab: Naive Bayes Classifier
**CSE303 Machine Learning | Unit III: Probability and Bayes Learning | Syllabus Exp 9 (naive Bayesian classifier and its accuracy)**

Pattern: **demo → practice → predict-then-run → deliberate error → reflection**.

| Part | Topic | Track |
|---|---|---|
| 1 | Probability and Bayes' theorem | core |
| 2 | Naive Bayes by hand (Play Tennis), and the zero-frequency error | core |
| 3 | Gaussian Naive Bayes in sklearn, log-probabilities, and scaling | core |
| 4 | Mini Project A: Naive Bayes on digits (continues Week 7/8/9) | image |
| 5 | Mini Project B: Naive Bayes on the six sentences (continues Week 7/8/9) | NLP |
| 6 | Reflection and capstone link | |

**Rule carried from Week 5:** split first; fit every scaler, PCA and tuned setting on **train only**.

In [65]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
np.set_printoptions(precision=4, suppress=True)

## Part 1: Probability and Bayes' Theorem
$$P(C\mid x)=\frac{P(x\mid C)\,P(C)}{P(x)}$$
**Posterior** = likelihood × prior / evidence. The evidence $P(x)=\sum_C P(x\mid C)P(C)$ is the same for every class, so to *compare* classes we only need the numerator.

### Predict-then-run
A disease affects 1% of people. A test is positive for 90% of sick people (sensitivity) and also for 5% of healthy people (false-positive rate). You test positive. Chance you are actually sick? Guess before running: **~90% / ~50% / ~15%**

In [66]:
prior, sens, fpr = 0.01, 0.90, 0.05           # P(sick), P(+|sick), P(+|healthy)
evidence = sens * prior + fpr * (1 - prior)    # P(+)
posterior = sens * prior / evidence            # P(sick | +)
print('P(positive)      =', round(evidence, 4))
print('P(sick | positive) =', round(posterior, 4))

# sanity check by simulation: 1,000,000 people
rng = np.random.default_rng(1); n = 1_000_000
sick = rng.random(n) < prior
pos = np.where(sick, rng.random(n) < sens, rng.random(n) < fpr)
print('simulated          =', round(sick[pos].mean(), 4), f'({sick[pos].sum()} sick among {pos.sum()} positives)')

P(positive)      = 0.0585
P(sick | positive) = 0.1538
simulated          = 0.1545 (9035 sick among 58469 positives)


### Student Practice 1
An email filter knows: 40% of mail is spam; the word *free* appears in 50% of spam and 5% of non-spam.
1. Compute $P(\text{spam}\mid \text{free})$ by hand, then in code.
2. If the spam rate were only 5%, what would the posterior be? Why does it drop so much?
3. In the disease example, which term of Bayes' theorem do people usually ignore?

In [67]:
spam_prior = 0.40
p_free_spam = 0.50
p_free_nonspam = 0.05

evidence = (p_free_spam * spam_prior) + (p_free_nonspam * (1 - spam_prior))

posterior = (p_free_spam * spam_prior) / evidence

print("P(spam | free) =", round(posterior, 4))




spam_prior = 0.05

evidence = (p_free_spam * spam_prior) + (p_free_nonspam * (1 - spam_prior))

posterior = (p_free_spam * spam_prior) / evidence

print("P(spam | free), 5% spam rate =", round(posterior, 4))




print("The prior probability P(spam) is commonly ignored.")


P(spam | free) = 0.8696
P(spam | free), 5% spam rate = 0.3448
The prior probability P(spam) is commonly ignored.


## Part 2: Naive Bayes by Hand (Play Tennis)
Classic 14-day dataset (Mitchell, Ch. 6). Predict **Play** from four categorical features.

The **naive** assumption: features are independent *given the class*, so
$$P(x_1,\dots,x_d\mid C)=\prod_{j=1}^{d}P(x_j\mid C)\quad\Rightarrow\quad \hat C=\arg\max_C\; P(C)\prod_j P(x_j\mid C)$$

Steps: (1) class priors, (2) conditional probability tables from counts, (3) multiply, (4) normalise, (5) pick the largest.

| Day | Outlook | Temp | Humidity | Wind | Play |
|---|---|---|---|---|---|
| 1 | Sunny | Hot | High | Weak | No |
| 2 | Sunny | Hot | High | Strong | No |
| 3 | Overcast | Hot | High | Weak | Yes |
| 4 | Rain | Mild | High | Weak | Yes |
| 5 | Rain | Cool | Normal | Weak | Yes |
| 6 | Rain | Cool | Normal | Strong | No |
| 7 | Overcast | Cool | Normal | Strong | Yes |
| 8 | Sunny | Mild | High | Weak | No |
| 9 | Sunny | Cool | Normal | Weak | Yes |
| 10 | Rain | Mild | Normal | Weak | Yes |
| 11 | Sunny | Mild | Normal | Strong | Yes |
| 12 | Overcast | Mild | High | Strong | Yes |
| 13 | Overcast | Hot | Normal | Weak | Yes |
| 14 | Rain | Mild | High | Strong | No |

In [68]:
rows = [
 ('Sunny','Hot','High','Weak','No'), ('Sunny','Hot','High','Strong','No'), ('Overcast','Hot','High','Weak','Yes'),
 ('Rain','Mild','High','Weak','Yes'), ('Rain','Cool','Normal','Weak','Yes'), ('Rain','Cool','Normal','Strong','No'),
 ('Overcast','Cool','Normal','Strong','Yes'), ('Sunny','Mild','High','Weak','No'), ('Sunny','Cool','Normal','Weak','Yes'),
 ('Rain','Mild','Normal','Weak','Yes'), ('Sunny','Mild','Normal','Strong','Yes'), ('Overcast','Mild','High','Strong','Yes'),
 ('Overcast','Hot','Normal','Weak','Yes'), ('Rain','Mild','High','Strong','No')]
F = np.array([r[:-1] for r in rows]); y = np.array([r[-1] for r in rows])
features = ['Outlook', 'Temp', 'Humidity', 'Wind']
print('Priors: Yes =', (y == 'Yes').sum(), '/ 14,  No =', (y == 'No').sum(), '/ 14')
for j, name in enumerate(features):
    print(name)
    for v in sorted(set(F[:, j])):
        print(f'   {v:9} Yes {((F[:, j] == v) & (y == "Yes")).sum()}/9    No {((F[:, j] == v) & (y == "No")).sum()}/5')

Priors: Yes = 9 / 14,  No = 5 / 14
Outlook
   Overcast  Yes 4/9    No 0/5
   Rain      Yes 3/9    No 2/5
   Sunny     Yes 2/9    No 3/5
Temp
   Cool      Yes 3/9    No 1/5
   Hot       Yes 2/9    No 2/5
   Mild      Yes 4/9    No 2/5
Humidity
   High      Yes 3/9    No 4/5
   Normal    Yes 6/9    No 1/5
Wind
   Strong    Yes 3/9    No 3/5
   Weak      Yes 6/9    No 2/5


In [69]:
def nb_scores(q, alpha=0):
    """Unnormalised score P(C) * prod_j P(x_j|C) and the normalised posterior. alpha = Laplace smoothing."""
    out = {}
    for c in ['Yes', 'No']:
        Fc = F[y == c]
        p = float((y == c).mean())
        for j, v in enumerate(q):
            k = len(set(F[:, j]))                                  # how many values this feature can take
            p *= float((Fc[:, j] == v).sum() + alpha) / (len(Fc) + alpha * k)
        out[c] = p
    total = sum(out.values())
    return out, {c: out[c] / total for c in out}

### Predict-then-run
Today is **(Sunny, Cool, High, Strong)**. The prior favours *Yes* (9/14). Will Naive Bayes say Yes or No? Guess, then run.

In [70]:
q = ('Sunny', 'Cool', 'High', 'Strong')
scores, post = nb_scores(q)
print('scores   :', {c: round(v, 5) for c, v in scores.items()})
print('posterior:', {c: round(v, 4) for c, v in post.items()})
print('prediction:', max(post, key=post.get))

scores   : {'Yes': 0.00529, 'No': 0.02057}
posterior: {'Yes': 0.2046, 'No': 0.7954}
prediction: No


### Student Practice 2 (do on paper first, as in an exam)
For today = (Sunny, Cool, High, Strong):
1. Write $P(\text{Yes})\,P(\text{Sunny}|\text{Yes})\,P(\text{Cool}|\text{Yes})\,P(\text{High}|\text{Yes})\,P(\text{Strong}|\text{Yes})$ as a product of fractions, then evaluate it. Do the same for *No*.
2. Normalise: $P(\text{Yes}\mid x)$ = ___ %
3. Redo it with Laplace smoothing ($\alpha=1$). Does the prediction change? Does the probability?
4. **Accuracy on test days (Exp 9).** Using $\alpha=1$, classify these five unseen days and report the accuracy:

| Test day | Outlook | Temp | Humidity | Wind | True Play |
|---|---|---|---|---|---|
| T1 | Overcast | Mild | Normal | Weak | Yes |
| T2 | Rain | Hot | High | Strong | No |
| T3 | Sunny | Hot | Normal | Weak | Yes |
| T4 | Rain | Cool | High | Weak | Yes |
| T5 | Sunny | Mild | High | Strong | No |

5. Verify your function against sklearn's `CategoricalNB(alpha=1)` (hint: encode the strings with `OrdinalEncoder`).

In [71]:
tests = [(('Overcast','Mild','Normal','Weak'), 'Yes'), (('Rain','Hot','High','Strong'), 'No'),
         (('Sunny','Hot','Normal','Weak'), 'Yes'), (('Rain','Cool','High','Weak'), 'Yes'),
         (('Sunny','Mild','High','Strong'), 'No')]


q = ('Sunny', 'Cool', 'High', 'Strong')

scores, post = nb_scores(q)

print("Without smoothing:")
print("Scores:", scores)
print("Posterior:", post)
print("Prediction:", max(post, key=post.get))




scores_s, post_s = nb_scores(q, alpha=1)

print("\nWith Laplace smoothing:")
print("Scores:", scores_s)
print("Posterior:", post_s)
print("Prediction:", max(post_s, key=post_s.get))




predictions = []

for x, actual in tests:
    s, p = nb_scores(x, alpha=1)
    prediction = max(p, key=p.get)
    predictions.append(prediction)

actual = [label for x, label in tests]

accuracy = np.mean(np.array(predictions) == np.array(actual))

print("\nTest predictions:", predictions)
print("Actual labels:", actual)
print("Test accuracy:", accuracy)




encoder = OrdinalEncoder()

F_encoded = encoder.fit_transform(F)

test_encoded = encoder.transform(
    np.array([x for x, y in tests])
)

model = CategoricalNB(alpha=1)

model.fit(F_encoded, y)

sk_predictions = model.predict(test_encoded)

print("\nSklearn predictions:", sk_predictions)
print("Predictions match:", np.array_equal(sk_predictions, predictions))


Without smoothing:
Scores: {'Yes': 0.005291005291005291, 'No': 0.02057142857142857}
Posterior: {'Yes': 0.20458265139116202, 'No': 0.795417348608838}
Prediction: No

With Laplace smoothing:
Scores: {'Yes': 0.007083825265643448, 'No': 0.018221574344023325}
Posterior: {'Yes': 0.2799333492025708, 'No': 0.7200666507974292}
Prediction: No

Test predictions: ['Yes', 'No', 'Yes', 'Yes', 'No']
Actual labels: ['Yes', 'No', 'Yes', 'Yes', 'No']
Test accuracy: 1.0

Sklearn predictions: ['Yes' 'No' 'Yes' 'Yes' 'No']
Predictions match: True


### Deliberate error: the zero-frequency problem
Today is **(Overcast, Hot, High, Strong)**. *Overcast* never occurs with *No* in the training data. Run the cell, then explain in one line why $\alpha=0$ is dangerous.

In [72]:
q2 = ('Overcast', 'Hot', 'High', 'Strong')
for a in [0, 1]:
    s, p = nb_scores(q2, alpha=a)
    print(f'alpha={a}: scores', {c: round(v, 5) for c, v in s.items()}, ' posterior', {c: round(float(v), 3) for c, v in p.items()})

alpha=0: scores {'Yes': 0.00705, 'No': 0.0}  posterior {'Yes': 1.0, 'No': 0.0}
alpha=1: scores {'Yes': 0.00885, 'No': 0.00683}  posterior {'Yes': 0.564, 'No': 0.436}


## Part 3: Gaussian Naive Bayes (continuous features)
For numeric features, model each feature *within each class* as a bell curve with that class's mean $\mu_{jc}$ and variance $\sigma^2_{jc}$:
$$P(x_j\mid C)=\frac{1}{\sqrt{2\pi\sigma^2_{jc}}}\exp\!\Big(-\frac{(x_j-\mu_{jc})^2}{2\sigma^2_{jc}}\Big)$$
sklearn stores these as `theta_` (means) and `var_` (variances). In practice we add **log-probabilities** instead of multiplying (see the deliberate error below).

In [73]:
Xi, yi = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=1, stratify=yi)
gnb = GaussianNB().fit(Xi_tr, yi_tr)
print('class priors:', gnb.class_prior_)
print('class means (rows = setosa, versicolor, virginica):\n', gnb.theta_)
print('class variances:\n', gnb.var_)
print('test accuracy:', round(gnb.score(Xi_te, yi_te), 4))

class priors: [0.3333 0.3333 0.3333]
class means (rows = setosa, versicolor, virginica):
 [[5.0229 3.4343 1.4571 0.2429]
 [5.8714 2.78   4.2743 1.3229]
 [6.6771 2.9943 5.6371 2.0257]]
class variances:
 [[0.1275 0.132  0.0373 0.0076]
 [0.2015 0.0867 0.2042 0.0418]
 [0.3709 0.1125 0.3143 0.0831]]
test accuracy: 0.9778


### Student Practice 3
1. For the first test flower `x = Xi_te[0]`, compute the log-posterior of each class **yourself** from `gnb.class_prior_`, `gnb.theta_`, `gnb.var_` (Gaussian formula, summed over the 4 features). Which class wins? Does it match `gnb.predict`?
2. Repeat for every test row and check that your predictions equal `gnb.predict(Xi_te)`.
3. Print the confusion matrix. Which two species are confused?

In [74]:
# Q1-Q3: your code here

### Deliberate error: multiplying many small probabilities
Predict: what is the product of 200 probabilities, each equal to 0.01? Then run.

In [75]:
p = np.full(200, 0.01)
print('product     :', np.prod(p))
print('sum of logs :', np.sum(np.log(p)))

product     : 0.0
sum of logs : -921.0340371976183


### Predict-then-run
In Week 9, KNN needed scaling on Wine (unscaled PC1 was dominated by *proline*). Does standardising change Gaussian Naive Bayes? Guess: **helps a lot / no change / hurts**. Then run.

In [76]:
Xw, yw = load_wine(return_X_y=True)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=1, stratify=yw)
sw = StandardScaler().fit(Xw_tr)                       # fit on TRAIN only
Xw_tr_s, Xw_te_s = sw.transform(Xw_tr), sw.transform(Xw_te)
print('model            raw     scaled')
for name, mk in [('GaussianNB', lambda: GaussianNB()), ('KNN (K=3)', lambda: KNeighborsClassifier(3))]:
    a = mk().fit(Xw_tr, yw_tr).score(Xw_te, yw_te)
    b = mk().fit(Xw_tr_s, yw_tr).score(Xw_te_s, yw_te)
    print(f'{name:16} {a:.4f}  {b:.4f}')

model            raw     scaled
GaussianNB       0.9815  0.9815
KNN (K=3)        0.6667  0.9444


### Student Practice 4
1. Explain in one line why Gaussian NB does not care about feature units. (Look at $(x-\mu)^2/\sigma^2$: what happens to it if you change cm to mm?)
2. One split can be lucky. Report GaussianNB accuracy on the full Wine data with 5-fold cross-validation (mean and std). Why is it safe to skip the scaler here?

In [77]:
raw_pixels = Xd_train[:, Xd_train.std(axis=0) > 0]

raw_corr = np.corrcoef(raw_pixels, rowvar=False)

raw_off = raw_corr[
    ~np.eye(raw_corr.shape[0], dtype=bool)
]

print("Raw pixel correlation:",
      round(np.mean(np.abs(raw_off)), 4))


pca30 = PCA(n_components=30)

scores30 = pca30.fit_transform(Xtr_s)

pca_corr = np.corrcoef(scores30, rowvar=False)

pca_off = pca_corr[
    ~np.eye(pca_corr.shape[0], dtype=bool)
]

print("PCA correlation:",
      round(np.mean(np.abs(pca_off)), 4))



for k in [10, 20, 30, 40, 64]:

    pca = PCA(n_components=k)

    X_train_pca = pca.fit_transform(Xtr_s)

    X_test_pca = pca.transform(Xte_s)

    model = GaussianNB()

    model.fit(X_train_pca, yd_train)

    accuracy = model.score(X_test_pca, yd_test)

    print("PCA components:", k,
          "Accuracy:", round(accuracy, 4))


print("\nPCA changes the feature axes.")
print("GaussianNB assumes independent features.")
print("Therefore, even PCA(64) can change GaussianNB accuracy.")


Raw pixel correlation: 0.1257
PCA correlation: 0.0
PCA components: 10 Accuracy: 0.8296
PCA components: 20 Accuracy: 0.9148
PCA components: 30 Accuracy: 0.9259
PCA components: 40 Accuracy: 0.9222
PCA components: 64 Accuracy: 0.9185

PCA changes the feature axes.
GaussianNB assumes independent features.
Therefore, even PCA(64) can change GaussianNB accuracy.


# Mini Project A: Naive Bayes on Digits (Image Track — continuing from Weeks 7, 8 and 9)
Same split as before (`test_size=0.3, random_state=1, stratify`). Baselines you already have: **KNN (scaled, K=3) = 0.9796** and **Tree (entropy, full depth, unscaled) = 0.8815**. Question: how does Naive Bayes compare, and does Week 9's PCA help it?

In [78]:
digits = load_digits()
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=1, stratify=digits.target)
print('pixels with zero variance in train:', int((Xd_train.std(0) == 0).sum()), 'of 64')
print('GaussianNB test accuracy:', round(GaussianNB().fit(Xd_train, yd_train).score(Xd_test, yd_test), 4))

pixels with zero variance in train: 3 of 64
GaussianNB test accuracy: 0.8315


### Predict-then-run
Before running the cell above, guess: will GaussianNB score **above both** baselines, **between** them, or **below both**? The 64 pixels are clearly *not* independent given the digit. Is that a problem for NB?

### Student Practice 5
1. Fit `MultinomialNB()` (treat pixel intensity as a count) and `BernoulliNB(binarize=8)` (ink / no ink) on the raw pixels. Report test accuracy for both.
2. GaussianNB has one knob, `var_smoothing`. For values `[1e-9, 1e-3, 1e-2, 1e-1, 1]`, use 5-fold cross-validation **on the training set only**, pick the best, then report its test accuracy **once**. (Why not pick the value with the best test accuracy?)
3. Using that best model, which two digits are confused most often?

In [79]:


def gaussian_log_posterior(x, model):

    log_scores = np.log(model.class_prior_) - 0.5 * np.sum(
        np.log(2 * np.pi * model.var_) +
        ((x - model.theta_) ** 2) / model.var_,
        axis=1
    )

    return log_scores


x = Xi_te[0]

log_scores = gaussian_log_posterior(x, gnb)

print("Manual log posterior:", log_scores)

manual_class = gnb.classes_[np.argmax(log_scores)]

print("Manual prediction:", manual_class)
print("Sklearn prediction:", gnb.predict([x])[0])




manual_predictions = np.array([
    gnb.classes_[np.argmax(gaussian_log_posterior(row, gnb))]
    for row in Xi_te
])

sklearn_predictions = gnb.predict(Xi_te)

print("\nAll predictions match:",
      np.array_equal(manual_predictions, sklearn_predictions))




cm = confusion_matrix(yi_te, sklearn_predictions)

print("\nConfusion Matrix:")
print(cm)

print("Class order:", gnb.classes_)


Manual log posterior: [-491.155   -17.3865   -2.944 ]
Manual prediction: 2
Sklearn prediction: 2

All predictions match: True

Confusion Matrix:
[[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]
Class order: [0 1 2]


### Deliberate error: negative inputs
Since Week 9 we standardise pixels before PCA. Run the cell: what breaks, and why?

In [80]:
sc = StandardScaler().fit(Xd_train)                    # fit on TRAIN only
Xtr_s, Xte_s = sc.transform(Xd_train), sc.transform(Xd_test)
try:
    MultinomialNB().fit(Xtr_s, yd_train)
except ValueError as e:
    print('ValueError:', e)
print('GaussianNB on scaled pixels:', round(GaussianNB().fit(Xtr_s, yd_train).score(Xte_s, yd_test), 4), ' (raw pixels gave 0.8315)')

ValueError: Negative values in data passed to MultinomialNB (input X).
GaussianNB on scaled pixels: 0.7759  (raw pixels gave 0.8315)


### Link to Week 9: does decorrelating the pixels help?
Naive Bayes assumes features are independent given the class. PCA scores are *uncorrelated by construction*. Question: does PCA before NB help?

### Student Practice 6
1. Compute the mean absolute off-diagonal correlation (a) of the non-constant raw pixels and (b) of the first 30 PCA scores of `Xtr_s`.
2. For k in `[10, 20, 30, 40, 64]` fit `PCA(k)` on `Xtr_s` (train only), then `GaussianNB` on the scores, and report test accuracy.
3. `PCA(64)` is only a rotation of the scaled pixels: no information is lost. So why does its accuracy differ from GaussianNB on scaled pixels (above)?

In [81]:



mnb = MultinomialNB()

mnb.fit(Xd_train, yd_train)

print("MultinomialNB accuracy:",
      round(mnb.score(Xd_test, yd_test), 4))



bnb = BernoulliNB(binarize=8)

bnb.fit(Xd_train, yd_train)

print("BernoulliNB accuracy:",
      round(bnb.score(Xd_test, yd_test), 4))



values = [1e-9, 1e-3, 1e-2, 1e-1, 1]

cv_means = []

for value in values:

    model = GaussianNB(var_smoothing=value)

    scores = cross_val_score(
        model,
        Xd_train,
        yd_train,
        cv=5
    )

    cv_means.append(scores.mean())

    print("Value:", value,
          "CV accuracy:", round(scores.mean(), 4))


best_value = values[np.argmax(cv_means)]

print("\nBest var_smoothing:", best_value)




best_model = GaussianNB(var_smoothing=best_value)

best_model.fit(Xd_train, yd_train)

predictions = best_model.predict(Xd_test)

print("Best model test accuracy:",
      round(best_model.score(Xd_test, yd_test), 4))




cm = confusion_matrix(yd_test, predictions)

cm_off = cm.copy()

np.fill_diagonal(cm_off, 0)

i, j = np.unravel_index(
    np.argmax(cm_off),
    cm_off.shape
)

print("\nMost confused digits:", i, "and", j)

print("Number of misclassifications:", cm_off[i, j])

print("\nConfusion Matrix:")
print(cm)


MultinomialNB accuracy: 0.8963
BernoulliNB accuracy: 0.8963
Value: 1e-09 CV accuracy: 0.8187
Value: 0.001 CV accuracy: 0.903
Value: 0.01 CV accuracy: 0.9173
Value: 0.1 CV accuracy: 0.9141
Value: 1 CV accuracy: 0.9046

Best var_smoothing: 0.01
Best model test accuracy: 0.9296

Most confused digits: 1 and 8
Number of misclassifications: 6

Confusion Matrix:
[[52  0  0  0  2  0  0  0  0  0]
 [ 0 44  4  0  0  0  0  0  6  1]
 [ 0  0 51  1  0  0  0  0  1  0]
 [ 0  0  0 50  0  0  0  2  2  1]
 [ 0  0  0  0 50  0  0  3  1  0]
 [ 0  0  0  1  0 53  1  0  0  0]
 [ 0  0  0  0  2  0 52  0  0  0]
 [ 0  0  0  0  1  0  0 52  1  0]
 [ 0  4  0  0  0  1  0  1 46  0]
 [ 0  0  1  0  0  0  0  1  0 52]]


# Mini Project B: Naive Bayes on the Six Sentences (NLP Track — continuing from Weeks 7, 8 and 9)
Same sentences and bag-of-words vectors. Naive Bayes is the classic text classifier (spam filters!). In Week 9, PCA failed to find the topics because *the* and *and* dominated the variance. Question: does NB do better?

In [82]:
sentences = [
    'the cricket team won the match',
    'the batsman scored a century today',
    'chop the onions and boil the pasta',
    'add salt and simmer the curry',
    'the new phone has a faster processor',
    'the laptop battery lasts all day',
]
labels = ['sports', 'sports', 'cooking', 'cooking', 'tech', 'tech']
vocab = sorted(set(w for s in sentences for w in s.lower().split()))
def vec(s): return [s.lower().split().count(w) for w in vocab]       # words outside the vocabulary are ignored
X_text = np.array([vec(s) for s in sentences])
print('matrix shape (docs x vocab):', X_text.shape)
nb_text = MultinomialNB(alpha=1).fit(X_text, labels)
print('classes:', nb_text.classes_, ' priors:', np.exp(nb_text.class_log_prior_))

matrix shape (docs x vocab): (6, 29)
classes: ['cooking' 'sports' 'tech']  priors: [0.3333 0.3333 0.3333]


### Predict-then-run
Multinomial NB with smoothing uses $P(w\mid C)=\dfrac{\text{count}(w,C)+\alpha}{\text{total words in }C+\alpha\,|V|}$ with $|V|=29$.
(a) How many word tokens are in the *sports* class (two sentences)? (b) Which class will NB give to **'the new curry'**? Is it confident?

### Student Practice 7
1. Predict the five sentences below with `nb_text` (`alpha=1`) and print the class probabilities. Accuracy?
   `'the team scored a century'` (sports), `'boil the pasta with salt'` (cooking), `'the phone battery lasts a day'` (tech), `'the match today'` (sports), `'the new curry'` (cooking)
2. By hand: for `'the team scored a century'` compute the three class scores from the count formula above and normalise. Compare with `predict_proba`.
3. Print the 5 words with the highest $P(w\mid C)$ in each class. Is *the* near the top everywhere? Why does that **not** hurt Naive Bayes, when it did hurt PCA?
4. Why is the probability for **'the new curry'** so spread out?

In [83]:


practice_sentences = [
    'the team scored a century',
    'boil the pasta with salt',
    'the phone battery lasts a day',
    'the match today',
    'the new curry'
]

actual_labels = [
    'sports',
    'cooking',
    'tech',
    'sports',
    'cooking'
]

X_practice = np.array([
    vec(s) for s in practice_sentences
])

predictions = nb_text.predict(X_practice)

probabilities = nb_text.predict_proba(X_practice)

for sentence, prediction, probability in zip(
    practice_sentences,
    predictions,
    probabilities
):

    print("\nSentence:", sentence)

    print("Prediction:", prediction)

    print("Probabilities:",
          dict(zip(nb_text.classes_, probability.round(4))))


accuracy = np.mean(
    np.array(predictions) == np.array(actual_labels)
)

print("\nAccuracy:", accuracy)




words = 'the team scored a century'.split()

manual_scores = {}

for c in nb_text.classes_:

    class_docs = [
        sentences[i]
        for i, label in enumerate(labels)
        if label == c
    ]

    class_words = ' '.join(class_docs).split()

    total_words = len(class_words)

    score = len(class_docs) / len(sentences)

    for word in words:

        count = class_words.count(word)

        score *= (count + 1) / (total_words + len(vocab))

    manual_scores[c] = score


total = sum(manual_scores.values())

manual_posterior = {
    c: score / total
    for c, score in manual_scores.items()
}

print("\nManual class scores:", manual_scores)

print("Manual posterior:", manual_posterior)



for index, c in enumerate(nb_text.classes_):

    top_indices = np.argsort(
        nb_text.feature_log_prob_[index]
    )[::-1][:5]

    print("\nClass:", c)

    for i in top_indices:

        print(vocab[i],
              round(np.exp(nb_text.feature_log_prob_[index, i]), 4))




print("\nThe new curry contains words associated")
print("with different topics, so the probabilities")
print("are distributed across the three classes.")



Sentence: the team scored a century
Prediction: sports
Probabilities: {np.str_('cooking'): np.float64(0.0487), np.str_('sports'): np.float64(0.8783), np.str_('tech'): np.float64(0.073)}

Sentence: boil the pasta with salt
Prediction: cooking
Probabilities: {np.str_('cooking'): np.float64(0.8121), np.str_('sports'): np.float64(0.1118), np.str_('tech'): np.float64(0.0761)}

Sentence: the phone battery lasts a day
Prediction: tech
Probabilities: {np.str_('cooking'): np.float64(0.0366), np.str_('sports'): np.float64(0.0846), np.str_('tech'): np.float64(0.8788)}

Sentence: the match today
Prediction: sports
Probabilities: {np.str_('cooking'): np.float64(0.1653), np.str_('sports'): np.float64(0.7107), np.str_('tech'): np.float64(0.124)}

Sentence: the new curry
Prediction: cooking
Probabilities: {np.str_('cooking'): np.float64(0.4372), np.str_('sports'): np.float64(0.235), np.str_('tech'): np.float64(0.3279)}

Accuracy: 1.0

Manual class scores: {np.str_('cooking'): 1.020216362589139e-08, n

### Deliberate error: almost no smoothing
`'the cricket phone'` mixes a sports word with a tech word. Run the cell and compare how *confident* each model is.

In [84]:
for a in [1e-10, 1.0]:
    m = MultinomialNB(alpha=a).fit(X_text, labels)
    print(f'alpha={a:<6}', dict(zip(m.classes_.tolist(), m.predict_proba([vec('the cricket phone')])[0].round(3).tolist())))

alpha=1e-10  {'cooking': 0.0, 'sports': 0.656, 'tech': 0.344}
alpha=1.0    {'cooking': 0.215, 'sports': 0.462, 'tech': 0.323}


## Part 6: Reflection (write 2–3 lines each)
1. In your own words: what is the "naive" assumption, and what does it make easy?
2. What is the zero-frequency problem? Show it with the Play Tennis example and say how Laplace smoothing fixes it.
3. Why do real implementations add log-probabilities instead of multiplying probabilities?
4. Wine: scaling changed KNN but not Gaussian NB. What does that tell you about how each model uses the features?
5. Digits: PCA helped NB but hurt the tree (Week 9). Explain using the independence assumption.
6. Naive Bayes is fast and needs little data, but its probabilities are often overconfident. When would you still choose it?
7. **Capstone link (Milestone, Week 10):** will Naive Bayes belong in your pipeline, at least as a baseline? If yes, add its accuracy to your comparison table in your repo's `notebooks/` folder; if no, justify.

*Submit this notebook to GitHub under `labs/week10_naive_bayes/`.*